In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__results__.html
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__output__.json
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/cleaned_HVA_dataset.xlsx
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.xlsx
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/custom.css


In [2]:
# =========================================
# Three independent binary classification experiments
#
# Experiment 1: Human vs GPT
# Experiment 2: Human vs Grok
# Experiment 3: Human vs Qwen
#
# No text preprocessing
# Grouped split by filename
# =========================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

warnings.filterwarnings("ignore")

# =========================================
# 1) Load already-preprocessed long dataset
# =========================================

DATA_PATH = (
    "/kaggle/input/notebooks/aabdollahii/"
    "8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv"
)

print("Data path exists:", os.path.exists(DATA_PATH))
print("Data path:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("\nOriginal dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

# =========================================
# 2) Validate and prepare columns
# No text preprocessing is applied
# =========================================

required_columns = ["filename", "source", "content"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

df = df.copy()

df["source"] = df["source"].astype(str).str.strip().str.lower()

valid_sources = ["human", "gpt", "grok", "qwen"]

df = df[
    df["source"].isin(valid_sources)
].copy()

# Remove only invalid text records.
# The text itself is not modified.
df["content"] = df["content"].fillna("").astype(str)
df = df[df["content"].str.strip().str.len() > 0].reset_index(drop=True)

print("\nDataset shape after validation:", df.shape)

print("\nSource distribution:")
print(df["source"].value_counts())

# =========================================
# 3) Build classifiers
# =========================================

def build_models(n_train_samples):
    # KNN cannot use more neighbors than available training samples.
    knn_neighbors = min(15, max(1, n_train_samples))

    return {
        "TFIDF+MultinomialNB": Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    sublinear_tf=True,
                    lowercase=False
                )
            ),
            (
                "classifier",
                MultinomialNB(alpha=1.0)
            )
        ]),

        "TFIDF+LinearSVC": Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    sublinear_tf=True,
                    lowercase=False
                )
            ),
            (
                "classifier",
                LinearSVC(
                    C=1.0,
                    random_state=42
                )
            )
        ]),

        "TFIDF+RandomForest": Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    sublinear_tf=True,
                    lowercase=False
                )
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=300,
                    random_state=42,
                    n_jobs=-1
                )
            )
        ]),

        "TFIDF+KNN": Pipeline([
            (
                "tfidf",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    min_df=2,
                    max_df=0.95,
                    sublinear_tf=True,
                    lowercase=False
                )
            ),
            (
                "classifier",
                KNeighborsClassifier(
                    n_neighbors=knn_neighbors,
                    metric="cosine"
                )
            )
        ])
    }

# =========================================
# 4) Evaluate one model
# =========================================

def evaluate_model(
    model,
    model_name,
    experiment_name,
    X_train,
    y_train,
    X_test,
    y_test
):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    y_score = None

    # Probability score for models that support predict_proba
    if hasattr(model, "predict_proba"):
        try:
            probabilities = model.predict_proba(X_test)

            if probabilities.ndim == 2 and probabilities.shape[1] == 2:
                y_score = probabilities[:, 1]
        except Exception:
            y_score = None

    # Decision score for LinearSVC
    if y_score is None and hasattr(model, "decision_function"):
        try:
            decision_scores = model.decision_function(X_test)

            if np.ndim(decision_scores) == 1:
                y_score = decision_scores
        except Exception:
            y_score = None

    accuracy = accuracy_score(y_test, y_pred)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_test,
            y_pred,
            average="binary",
            pos_label=1,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    )

    roc_auc = np.nan

    if y_score is not None and len(np.unique(y_test)) == 2:
        try:
            roc_auc = roc_auc_score(y_test, y_score)
        except Exception:
            roc_auc = np.nan

    print("\n" + "=" * 75)
    print(f"Experiment: {experiment_name}")
    print(f"Model: {model_name}")
    print("=" * 75)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    if not np.isnan(roc_auc):
        print(f"ROC-AUC  : {roc_auc:.4f}")
    else:
        print("ROC-AUC  : Not available")

    print("\nConfusion matrix:")
    display(
        pd.DataFrame(
            cm,
            index=["true_human", "true_machine"],
            columns=["pred_human", "pred_machine"]
        )
    )

    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1],
            target_names=["human", "machine"],
            zero_division=0
        )
    )

    result = {
        "experiment": experiment_name,
        "model": model_name,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "train_rows": len(y_train),
        "test_rows": len(y_test)
    }

    prediction_df = pd.DataFrame({
        "y_true": y_test,
        "y_pred": y_pred
    })

    if y_score is not None:
        prediction_df["score_machine"] = y_score

    return result, prediction_df, model

# =========================================
# 5) Run one experiment
# =========================================

def run_experiment(df, ai_source):
    experiment_name = f"human_vs_{ai_source}"

    print("\n\n")
    print("#" * 85)
    print(f"STARTING EXPERIMENT: {experiment_name}")
    print("#" * 85)

    # Select human and one AI source only
    experiment_df = df[
        df["source"].isin(["human", ai_source])
    ].copy()

    experiment_df["label"] = np.where(
        experiment_df["source"] == "human",
        0,
        1
    )

    experiment_df = experiment_df.reset_index(drop=True)

    print("\nExperiment dataset shape:", experiment_df.shape)

    print("\nSource distribution:")
    print(experiment_df["source"].value_counts())

    print("\nLabel distribution:")
    print(
        experiment_df["label"]
        .value_counts()
        .rename(index={0: "human", 1: ai_source})
    )

    # Group split by filename
    X = experiment_df["content"].values
    y = experiment_df["label"].values
    groups = experiment_df["filename"].values

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=groups)
    )

    train_df = experiment_df.iloc[train_idx].reset_index(drop=True)
    test_df = experiment_df.iloc[test_idx].reset_index(drop=True)

    X_train = train_df["content"].values
    y_train = train_df["label"].values

    X_test = test_df["content"].values
    y_test = test_df["label"].values

    print("\nTrain shape:", train_df.shape)
    print("Test shape :", test_df.shape)

    print("\nTrain source distribution:")
    print(train_df["source"].value_counts())

    print("\nTest source distribution:")
    print(test_df["source"].value_counts())

    # Confirm there is no filename leakage
    train_files = set(train_df["filename"])
    test_files = set(test_df["filename"])
    overlap = train_files.intersection(test_files)

    print("\nFilename overlap between train and test:", len(overlap))

    if len(overlap) == 0:
        print("Grouped split is clean.")
    else:
        print("Warning: filename leakage detected.")

    # Save experiment train/test files
    safe_experiment_name = experiment_name.replace(" ", "_")

    train_path = (
        f"/kaggle/working/{safe_experiment_name}_train.csv"
    )
    test_path = (
        f"/kaggle/working/{safe_experiment_name}_test.csv"
    )

    train_df.to_csv(
        train_path,
        index=False,
        encoding="utf-8-sig"
    )

    test_df.to_csv(
        test_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nSaved train split:", train_path)
    print("Saved test split :", test_path)

    # Train and evaluate models
    models = build_models(len(X_train))

    experiment_results = []
    experiment_predictions = {}

    for model_name, model in models.items():
        try:
            result, prediction_df, trained_model = evaluate_model(
                model=model,
                model_name=model_name,
                experiment_name=experiment_name,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test
            )

            experiment_results.append(result)
            experiment_predictions[model_name] = prediction_df

        except Exception as error:
            print(f"\nError in {model_name}:")
            print(error)

    # Save predictions for every model
    for model_name, prediction_df in experiment_predictions.items():
        safe_model_name = (
            model_name
            .replace("+", "_")
            .replace(" ", "_")
        )

        prediction_path = (
            f"/kaggle/working/"
            f"{safe_experiment_name}_{safe_model_name}_predictions.csv"
        )

        output_predictions = test_df.copy()
        output_predictions["y_true"] = prediction_df["y_true"].values
        output_predictions["y_pred"] = prediction_df["y_pred"].values

        if "score_machine" in prediction_df.columns:
            output_predictions["score_machine"] = (
                prediction_df["score_machine"].values
            )

        output_predictions.to_csv(
            prediction_path,
            index=False,
            encoding="utf-8-sig"
        )

        print("Saved predictions:", prediction_path)

    return experiment_results

# =========================================
# 6) Run all three experiments
# =========================================

all_results = []

for ai_source in ["gpt", "grok", "qwen"]:
    experiment_results = run_experiment(
        df=df,
        ai_source=ai_source
    )

    all_results.extend(experiment_results)

# =========================================
# 7) Compare all experiments
# =========================================

results_df = pd.DataFrame(all_results)

if len(results_df) > 0:
    results_df = results_df.sort_values(
        by=["experiment", "f1"],
        ascending=[True, False]
    ).reset_index(drop=True)

print("\n\nFinal results for all experiments:")
display(results_df)

results_path = "/kaggle/working/HVA_three_binary_experiments_results.csv"

results_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved combined results:", results_path)


Data path exists: True
Data path: /kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv

Original dataset shape: (5647, 7)
Columns: ['year', 'filename', 'word_count', 'content', 'source', 'label', 'n_words']


,year,filename,word_count,content,source,label,n_words
0,2003,HAM2-811011-027.ham,136,آغاز عملیات اجرایی سد جدید بر روی رودخانه کارو...,gpt,1,96
1,2003,HAM2-811011-027.ham,136,عملیات اجرایی بدنه و سرریز سد کارون ۴ که بلندت...,grok,1,92
2,2003,HAM2-811011-027.ham,136,آغاز ساخت بدنه و سرریز بلندترین سد کشور عملیات...,human,0,137
3,2003,HAM2-811011-027.ham,136,آغاز فازهای کلیدی ساخت بزرگ‌ترین سازه آبی کشور...,qwen,1,209
4,2003,HAM2-811014-088.ham,53,گزارش تازه آب ذخیره‌شده در سدهای تهران نشان می...,gpt,1,92



Dataset shape after validation: (5647, 7)

Source distribution:
source
grok     1519
human    1519
qwen     1519
gpt      1090
Name: count, dtype: int64



#####################################################################################
STARTING EXPERIMENT: human_vs_gpt
#####################################################################################

Experiment dataset shape: (2609, 7)

Source distribution:
source
human    1519
gpt      1090
Name: count, dtype: int64

Label distribution:
label
human    1519
gpt      1090
Name: count, dtype: int64

Train shape: (2087, 7)
Test shape : (522, 7)

Train source distribution:
source
human    1215
gpt       872
Name: count, dtype: int64

Test source distribution:
source
human    304
gpt      218
Name: count, dtype: int64

Filename overlap between train and test: 0
Grouped split is clean.

Saved train split: /kaggle/working/human_vs_gpt_train.csv
Saved test split : /kaggle/working/human_vs_gpt_test.csv

Experiment: human_vs_gpt
Model

,pred_human,pred_machine
true_human,304,0
true_machine,62,156



Classification report:
              precision    recall  f1-score   support

       human       0.83      1.00      0.91       304
     machine       1.00      0.72      0.83       218

    accuracy                           0.88       522
   macro avg       0.92      0.86      0.87       522
weighted avg       0.90      0.88      0.88       522


Experiment: human_vs_gpt
Model: TFIDF+LinearSVC
Accuracy : 0.9904
Precision: 0.9908
Recall   : 0.9862
F1-score : 0.9885
ROC-AUC  : 0.9997

Confusion matrix:


...


Classification report:
              precision    recall  f1-score   support

       human       0.96      1.00      0.98       304
     machine       1.00      0.95      0.97       304

    accuracy                           0.98       608
   macro avg       0.98      0.98      0.98       608
weighted avg       0.98      0.98      0.98       608


Experiment: human_vs_qwen
Model: TFIDF+KNN
Accuracy : 0.9211
Precision: 0.8879
Recall   : 0.9638
F1-score : 0.9243
ROC-AUC  : 0.9774

Confusion matrix:


,pred_human,pred_machine
true_human,267,37
true_machine,11,293



Classification report:
              precision    recall  f1-score   support

       human       0.96      0.88      0.92       304
     machine       0.89      0.96      0.92       304

    accuracy                           0.92       608
   macro avg       0.92      0.92      0.92       608
weighted avg       0.92      0.92      0.92       608

Saved predictions: /kaggle/working/human_vs_qwen_TFIDF_MultinomialNB_predictions.csv
Saved predictions: /kaggle/working/human_vs_qwen_TFIDF_LinearSVC_predictions.csv
Saved predictions: /kaggle/working/human_vs_qwen_TFIDF_RandomForest_predictions.csv
Saved predictions: /kaggle/working/human_vs_qwen_TFIDF_KNN_predictions.csv


Final results for all experiments:


,experiment,model,accuracy,precision,recall,f1,roc_auc,train_rows,test_rows
0,human_vs_gpt,TFIDF+LinearSVC,0.990421,0.990783,0.986239,0.988506,0.999713,2087,522
1,human_vs_gpt,TFIDF+RandomForest,0.938697,0.887500,0.977064,0.930131,0.990992,2087,522
2,human_vs_gpt,TFIDF+KNN,0.888889,0.900000,0.825688,0.861244,0.951805,2087,522
3,human_vs_gpt,TFIDF+MultinomialNB,0.881226,1.000000,0.715596,0.834225,0.997088,2087,522
4,human_vs_grok,TFIDF+RandomForest,0.898026,0.845714,0.973684,0.905199,0.970736,2430,608
5,human_vs_grok,TFIDF+LinearSVC,0.881579,0.946154,0.809211,0.872340,0.969735,2430,608
6,human_vs_grok,TFIDF+KNN,0.634868,0.910000,0.299342,0.450495,0.807290,2430,608
7,human_vs_grok,TFIDF+MultinomialNB,0.616776,1.000000,0.233553,0.378667,0.966121,2430,608
8,human_vs_qwen,TFIDF+LinearSVC,0.985197,0.990033,0.980263,0.985124,0.995293,2430,608
9,human_vs_qwen,TFIDF+RandomForest,0.975329,0.996564,0.953947,0.974790,0.995060,2430,608



Saved combined results: /kaggle/working/HVA_three_binary_experiments_results.csv
